In [1]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Item-Item Collaborative Filtering

## A sample example

In [2]:
data = {
    'User_1': [5, 3, 4, 4, 0, 0],
    'User_2': [3, 1, 2, 3, 3, 0],
    'User_3': [4, 3, 4, 3, 5, 0],
    'User_4': [3, 3, 3, 2, 4, 0],
    'User_5': [0, 1, 0, 1, 1, 5],
}

In [3]:
df_ratings = pd.DataFrame(data, index=['Item A', 'Item B', 'Item C', 'Item D', 'Item E', 'Item F'])
df_ratings

,User_1,User_2,User_3,User_4,User_5
Item A,5,3,4,3,0
Item B,3,1,3,3,1
Item C,4,2,4,3,0
Item D,4,3,3,2,1
Item E,0,3,5,4,1
Item F,0,0,0,0,5


## Calculating item similarity

The cosine_similarity() function calculates similarity between the rows of the input matrix.

It measures the cosine of the angle between two vectors (in this case, item rating vectors). A score close to 1 means the items were rated similarly by users.

In [4]:
item_similarity = cosine_similarity(df_ratings.values)
item_similarity

array([[1.        , 0.94284348, 0.98977827, 0.97980477, 0.74743385,
        0.        ],
       [0.94284348, 1.        , 0.96886393, 0.9217866 , 0.806079  ,
        0.18569534],
       [0.98977827, 0.96886393, 1.        , 0.95481983, 0.79321744,
        0.        ],
       [0.97980477, 0.9217866 , 0.95481983, 1.        , 0.73994007,
        0.16012815],
       [0.74743385, 0.806079  , 0.79321744, 0.73994007, 1.        ,
        0.14002801],
       [0.        , 0.18569534, 0.        , 0.16012815, 0.14002801,
        1.        ]])

In [5]:
df_item_similarity = pd.DataFrame(item_similarity,
                                  index=df_ratings.index,
                                  columns=df_ratings.index)

df_item_similarity

,Item A,Item B,Item C,Item D,Item E,Item F
Item A,1.000000,0.942843,0.989778,0.979805,0.747434,0.000000
Item B,0.942843,1.000000,0.968864,0.921787,0.806079,0.185695
Item C,0.989778,0.968864,1.000000,0.954820,0.793217,0.000000
Item D,0.979805,0.921787,0.954820,1.000000,0.739940,0.160128
Item E,0.747434,0.806079,0.793217,0.739940,1.000000,0.140028
Item F,0.000000,0.185695,0.000000,0.160128,0.140028,1.000000


In [6]:
similar_scores = df_item_similarity['Item A']
similar_scores

Item A    1.000000
Item B    0.942843
Item C    0.989778
Item D    0.979805
Item E    0.747434
Item F    0.000000
Name: Item A, dtype: float64

We will sort the scores in descending order and exclude the item itself (self-similarity is 1).

In [7]:
recommendations = similar_scores.sort_values(ascending=False)
recommendations = recommendations.drop('Item A')

In [8]:
recommendations

Item C    0.989778
Item D    0.979805
Item B    0.942843
Item E    0.747434
Item F    0.000000
Name: Item A, dtype: float64

In [9]:
def get_recommendations(item_name, similarity_matrix, num_recommendations=3):
    similar_scores = similarity_matrix[item_name]
    recommendations = similar_scores.sort_values(ascending=False)
    recommendations = recommendations.drop(item_name)
    return recommendations.head(num_recommendations)

In [10]:
item_c_recs = get_recommendations('Item C', df_item_similarity)
item_c_recs

Item A    0.989778
Item B    0.968864
Item D    0.954820
Name: Item C, dtype: float64

In [11]:
item_f_recs = get_recommendations('Item F', df_item_similarity)
item_f_recs

Item B    0.185695
Item D    0.160128
Item E    0.140028
Name: Item F, dtype: float64

# Matrix Factorisation (SVD - Singular Value Decomposition) with Surprise

**Matrix Factorisation** — a model-based approach that addresses the scalability and sparsity issues of standard Collaborative Filtering.

**Surprise library** - is specifically designed for building and analysing recommender systems.

In [12]:
ratings_data = [
    ('A', 'Movie_1', 5.0), ('A', 'Movie_2', 3.0), ('A', 'Movie_3', 4.0),
    ('B', 'Movie_1', 3.0), ('B', 'Movie_2', 1.0), ('B', 'Movie_4', 3.0),
    ('C', 'Movie_1', 4.0), ('C', 'Movie_3', 4.0), ('C', 'Movie_5', 5.0),
    ('D', 'Movie_2', 3.0), ('D', 'Movie_4', 2.0), ('D', 'Movie_5', 4.0),
    ('E', 'Movie_2', 1.0), ('E', 'Movie_4', 1.0), ('E', 'Movie_6', 5.0),
    ('F', 'Movie_3', 2.0), ('F', 'Movie_6', 4.0),
]

In [13]:
df = pd.DataFrame(ratings_data, columns=['user', 'item', 'rating'])
df

,user,item,rating
0,A,Movie_1,5.0
1,A,Movie_2,3.0
2,A,Movie_3,4.0
3,B,Movie_1,3.0
4,B,Movie_2,1.0
5,B,Movie_4,3.0
6,C,Movie_1,4.0
7,C,Movie_3,4.0
8,C,Movie_5,5.0
9,D,Movie_2,3.0


A Reader object is needed to parse the data structure.

Install scikit-surprise.

In [14]:
from surprise import Dataset, Reader

By specifying the rating_scale from 1 to 5, we ensure that the model knows the minimum and maximum possible values. It also, provides the necessary structure to validate and manage the data before it's passed to the algorithms.

In [15]:
reader = Reader(rating_scale=(1, 5))

In [16]:
reader

The Dataset object is the validated, pre-processed container that holds the rating information in an optimised format for the recommendation algorithms. It stores the ratings in a sparse internal format that is optimised for fast iteration and computation by Matrix Factorization algorithms like SVD.

In [17]:
data = Dataset.load_from_df(df, reader)

In [18]:
data

## Split the data into a training set and a testing set.

In [19]:
from surprise.model_selection import cross_validate, train_test_split

In [20]:
trainset, testset = train_test_split(data, test_size=0.25, random_state=42)

In [21]:
trainset

## Model Training

https://surprise.readthedocs.io/en/stable/matrix_factorization.html

In [22]:
from surprise import SVD

In [23]:
model = SVD(n_epochs=20, n_factors=5, random_state=42)

- **n_epochs**(default 20) - training iterations - represents the number of times the model will iterate over the entire training dataset, using an optimisation technique called Stochastic Gradient Descent. In each epoch, the model iterates through every known rating, calculates the prediction error, and slightly adjusts the user and item latent factors to reduce that error.
    - **too low(e.g., 5)**: the model might not learn the complex patterns, resulting in underfitting)
    - **too high(e.g., 100)**: the model might memorise the training data too well, resulting in overfitting.

- **n_factors**(default 100) - number of latent factors - defines the dimensionality of the feature space. It's the size of the User Vector and Item Vector.
    - **too low(e.g., 2)**: The model is too simple and cannot capture all the meaningful preferences, leading to underfitting.
    - **too high(e.g., 100)**: The model captures noise, leading to overfitting and higher computational cost.
- **random_state** - SVD starts by giving every user vector and item vector random, small values.

In [24]:
model.fit(trainset)

## Model Evaluation

In [25]:
predictions = model.test(testset)

In [26]:
predictions

[Prediction(uid='C', iid='Movie_3', r_ui=4.0, est=np.float64(3.417290350619325), details={'was_impossible': False}),
 Prediction(uid='D', iid='Movie_4', r_ui=2.0, est=np.float64(3.151343169622086), details={'was_impossible': False}),
 Prediction(uid='E', iid='Movie_2', r_ui=1.0, est=np.float64(3.0132249812627068), details={'was_impossible': False}),
 Prediction(uid='B', iid='Movie_1', r_ui=3.0, est=np.float64(3.303142619927674), details={'was_impossible': False}),
 Prediction(uid='C', iid='Movie_1', r_ui=4.0, est=np.float64(3.6138350150957863), details={'was_impossible': False})]

- uid - User ID - the unique identifier of the user
- iid - Item ID - the unique identifier of the item
- r_ui - True Rating - the actual, known rating the user gave to this item. The Ground Truth value taken from the test set.
- est - Estimated Rating - The predicted rating.
- details - Prediction Details - a dictionary containing medadata about the prediction. The details dictionary almost always reports: 'was_impossible': False, indicating that the prediction was successful, relying on the learned latent factor matrices.

In [27]:
from surprise import accuracy

In [28]:
rmse = accuracy.rmse(predictions, verbose=True)

RMSE: 1.0917


## Making Recommendations

In [29]:
user_id = 'A'
item_id_to_predict = 'Movie_5'

In [30]:
predicted_rating = model.predict(user_id, item_id_to_predict)

In [31]:
predicted_rating.est

np.float64(3.7227520665860934)

## Generate Top N Recommendations

In [32]:
def get_top_n_recommendations(predictions, n=3):
    top_n = {}
    for uid, iid, true_r, est, details in predictions:
        if uid not in top_n:
            top_n[uid] = []
            top_n[uid].append((iid, est))


    for uid, user_ratings in top_n.items():
        user_ratings.sort(key=lambda x: x[1], reverse=True)
        top_n[uid] = user_ratings[:n]   

    return top_n

In [33]:
top_recommendations = get_top_n_recommendations(predictions, n=2)

In [34]:
for uid, recs in top_recommendations.items():
    print(f"User {uid}: {recs}")

User C: [('Movie_3', np.float64(3.417290350619325))]
User D: [('Movie_4', np.float64(3.151343169622086))]
User E: [('Movie_2', np.float64(3.0132249812627068))]
User B: [('Movie_1', np.float64(3.303142619927674))]
